84(10) = 1010100(2)

00 - безадресные команды

1 - архитектура фон Неймана

0 - сумма элементов массива

In [ ]:
from IPython.display import display
import ipywidgets as widgets

class CPU:
    def __init__(self, ram_size=512):
        self.ram_size = ram_size
        self.reset()

    def reset(self):
        self.RAM = [0] * self.ram_size
        self.PC = 0
        self.IR = None
        self.STACK = []
        self.FLAGS = {"Z": 0}
        self.running = True

    def step(self):
        if not self.running:
            return

        self.IR = self.RAM[self.PC]
        self.PC += 1

        opcode = self.IR[0]
        arg = self.IR[1] if len(self.IR) > 1 else None

        if opcode == "PUSH":
            self.STACK.append(arg)

        elif opcode == "LOAD":
            if arg is None:
                addr = self.STACK.pop()
                self.STACK.append(self.RAM[addr])
            else:
                self.STACK.append(self.RAM[arg])

        elif opcode == "STORE":
            self.RAM[arg] = self.STACK.pop()

        elif opcode == "ADD":
            b = self.STACK.pop()
            a = self.STACK.pop()
            res = a + b
            self.STACK.append(res)
            self.FLAGS["Z"] = int(res == 0)

        elif opcode == "MUL":
            b = self.STACK.pop()
            a = self.STACK.pop()
            res = a * b
            self.STACK.append(res)
            self.FLAGS["Z"] = int(res == 0)

        elif opcode == "SUB":
            b = self.STACK.pop()
            a = self.STACK.pop()
            res = a - b
            self.STACK.append(res)
            self.FLAGS["Z"] = int(res == 0)

        elif opcode == "JMP":
            self.PC = arg

        elif opcode == "JZ":
            if self.FLAGS["Z"] == 1:
                self.PC = arg

        elif opcode == "CMP":
            b = self.STACK.pop()
            a = self.STACK.pop()
            self.FLAGS["Z"] = int(a == b)

        elif opcode == "HALT":
            self.running = False


def assemble(program):
    labels = {}
    code = []
    pc = 0

    # Первый проход — метки
    for line in program:
        if ":" in line:
            labels[line.replace(":", "")] = pc
        else:
            pc += 1

    # Второй проход — инструкции
    for line in program:
        if ":" in line:
            continue

        parts = line.split()

        # инструкция без операнда
        if len(parts) == 1:
            code.append((parts[0],))

        # инструкция с операндом
        else:
            arg = parts[1]
            arg = int(arg) if arg.lstrip("-").isdigit() else labels[arg]
            code.append((parts[0], arg))

    return code


cpu = CPU()

pc_label = widgets.Label()
ir_label = widgets.Label()
flags_label = widgets.Label()

stack_box = widgets.Textarea(
    description="STACK",
    layout=widgets.Layout(width="200px", height="200px")
)

ram_box = widgets.Textarea(
    description="RAM (0–20)",
    layout=widgets.Layout(width="300px", height="200px")
)

def update_gui():
    pc_label.value = f"PC: {cpu.PC}"
    ir_label.value = f"IR: {cpu.IR}"
    flags_label.value = f"FLAGS Z={cpu.FLAGS['Z']}"
    stack_box.value = "\n".join(map(str, reversed(cpu.STACK)))
    ram_box.value = "\n".join(f"{i}: {cpu.RAM[i]}" for i in range(20))

def step_clicked(b):
    cpu.step()
    update_gui()

def run_clicked(b):
    while cpu.running:
        cpu.step()
    update_gui()

def reset_clicked(b):
    cpu.reset()
    load_program()
    update_gui()

step_btn = widgets.Button(description="STEP")
run_btn = widgets.Button(description="RUN")
reset_btn = widgets.Button(description="RESET")

step_btn.on_click(step_clicked)
run_btn.on_click(run_clicked)
reset_btn.on_click(reset_clicked)

display(widgets.VBox([
    widgets.HBox([pc_label, ir_label, flags_label]),
    widgets.HBox([stack_box, ram_box]),
    widgets.HBox([step_btn, run_btn, reset_btn])
]))


program = [
    "PUSH 0",
    "STORE 190",

    "PUSH 0",
    "STORE 191",

"LOOP:",
    "LOAD 191",
    "LOAD 199",
    "CMP",
    "JZ END",

    "LOAD 190",
    "PUSH 200",
    "LOAD 191",
    "ADD",
    "LOAD",
    "ADD",
    "STORE 190",

    "LOAD 191",
    "PUSH 1",
    "ADD",
    "STORE 191",

    "JMP LOOP",

"END:",
    "LOAD 190",
    "HALT"
]

def load_program():
    code = assemble(program)
    cpu.RAM[:len(code)] = code

    cpu.RAM[200:210] = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    cpu.RAM[199] = 10

load_program()
update_gui()
